In [1]:
import pandas as pd 

In [2]:
df = pd.read_csv('enrolment_1.csv')

# 과제 조건

### 2,000명의 코드잇 대학교 학생들이 수강신청을 했습니다.

수강신청에는 다음 3개의 조건이 있습니다.

1. "information technology" 과목은 심화과목이라 1학년은 수강할 수 없습니다.
2. "commerce" 과목은 기초과목이고 많은 학생들이 듣는 수업이라 4학년은 수강할 수 없습니다.
3. 수강생이 5명이 되지 않으면 강의는 폐강되어 수강할 수 없습니다.

기존 DataFrame에 "status"라는 이름의 column을 추가하고, 학생이 수강 가능한 상태이면 "allowed", 수강 불가능한 상태이면 "not allowed"를 넣어주세요.

In [3]:
df["status"] = "allowed"

In [4]:
df

,id,year,course name,status
0,2777729,1,information technology,allowed
1,2777730,2,science,allowed
2,2777765,1,arts,allowed
3,2777766,2,arts,allowed
4,2777785,1,mba,allowed
...,...,...,...,...
1995,2796805,3,computer application,allowed
1996,2796812,1,nursing,allowed
1997,2796813,2,nursing,allowed
1998,2796814,3,nursing,allowed


In [5]:
# 조건1 
c1 = ((df["course name"] == "information technology") & (df["year"] == 1))

In [6]:
df.loc[c1, "status"] = "not allowed"

In [7]:
# 조건2 
c2 = ((df["course name"] == "commerce") & (df["year"] == 4))

In [8]:
df.loc[c2, "status"] = "not allowed"

In [9]:
# 조건3
# "status"가 "allowed"인 학생들만 필터링
# "course name"만 선택한 후, value_counts()를 사용해서 각 과목별 수강생 수를 계산
course_counts = df[df["status"] == "allowed"]["course name"].value_counts()

In [10]:
course_counts

course name
arts                     158
science                  123
commerce                 101
english                   56
education                 41
                        ... 
environmental science      1
jmc                        1
biological sciences        1
aqua culture               1
dcl                        1
Name: count, Length: 296, dtype: int64

In [11]:
closed_courses = course_counts[course_counts < 5].index
# .index를 사용하면, 수강생이 5명 미만인 과목의 "이름"만 가져올 수 있음

In [12]:
df.loc[df["course name"].isin(closed_courses), "status"] = "not allowed"
# .isin(): df["course name"].isin(closed_courses)
# → "course name"이 closed_courses 목록에 있는지 확인하는 조건.
# 조건을 만족한다면 isin()함수와 .loc[] 함수를 이용해 "status"를 "not allowed"로 변경하면 폐강 처리가 완료된다. 

In [13]:
df

,id,year,course name,status
0,2777729,1,information technology,not allowed
1,2777730,2,science,allowed
2,2777765,1,arts,allowed
3,2777766,2,arts,allowed
4,2777785,1,mba,allowed
...,...,...,...,...
1995,2796805,3,computer application,allowed
1996,2796812,1,nursing,allowed
1997,2796813,2,nursing,allowed
1998,2796814,3,nursing,allowed


# 과제 2 

## 과제 조건 

### 수강 신청이 완료되었습니다. 이제 각 과목을 수강하는 학생수에 따라 크기가 다른 강의실을 배치하려고 합니다.

강의실은 규모에 따라 "Auditorium", "Large room", "Medium room", "Small room" 총 4가지 종류가 있습니다.

아래 조건에 따라 강의실 종류를 지정해 주세요.

1. 80명 이상의 학생이 수강하는 과목은 "Auditorium"에서 진행됩니다. 
2. 40명 이상, 80명 미만의 학생이 수강하는 과목은 "Large room"에서 진행됩니다. 
3. 15명 이상, 40명 미만의 학생이 수강하는 과목은 "Medium room"에서 진행됩니다. 
4. 5명 이상, 15명 미만의 학생이 수강하는 과목은 "Small room"에서 진행됩니다. 
5. 폐강 등의 이유로 status가 "not allowed"인 수강생은 room assignment 또한 "not assigned"가 되어야 합니다.

In [15]:
# 수강 허용된 학생만 고려하여 과정별 학생 수 계산
course_counts = df[df["status"] == "allowed"]["course name"].value_counts()

In [16]:
# 강의실 배정 함수 정의
# .get(course, 0): .get(key, default_value)의 의미는 key가 존재하면 해당 값을 반환하고, 존재하지 않으면 
# default_value(기본값)를 반환하는 함수,  
def assign_room(course):
    if course_counts.get(course, 0) >= 80:
        return "Auditorium"
    elif course_counts.get(course, 0) >= 40:
        return "Large room"
    elif course_counts.get(course, 0) >= 15:
        return "Medium room"
    elif course_counts.get(course, 0) >= 5:
        return "Small room"
    else:
        return "not assigned"

# `.get()`을 쓰는 이유 

### 만약 .get()을 사용하지 않고 아래처럼 직접 접근하면?


`course_counts[course]`  # 과목이 없으면 KeyError 발생!
➡ **KeyError(키 오류)**가 발생할 수 있어! 😵

하지만 .get()을 사용하면:

`course_counts.get(course, 0)`  # 과목이 없으면 0을 반환!
➡ 해당 과목이 없으면 자동으로 0을 반환해서 오류를 방지할 수 있어! 🎯

예제 코드로 이해해보자!

`import pandas as pd`

### 예제 데이터프레임 생성
`data = {'course name': ['math', 'science', 'history', 'math', 'science', 'math']}`  
`df = pd.DataFrame(data)`  

### 각 과목별 학생 수 계산
`course_counts = df["course name"].value_counts()`  
`print(course_counts)`

출력 결과:

math       3  
science    2  
history    1  
Name: course name, dtype: int64  

이제 존재하는 과목과 존재하지 않는 과목을 비교해볼게.  

`print(course_counts.get("math", 0))`      # 존재하는 과목 → 3 반환  
`print(course_counts.get("history", 0))`   # 존재하는 과목 → 1 반환  
`print(course_counts.get("english", 0))`   # 존재하지 않는 과목 → 0 반환  

출력 결과:  
  
3  
1   
0  
📌 "english"는 `course_counts`에 없지만 `.get("english", 0)`을 사용했기 때문에 KeyError 없이 0이 반환됨!

4️⃣ .get()을 안 쓰면 어떻게 될까?  
만약 .get() 없이 아래처럼 직접 접근하면?  

`print(course_counts["english"])`  
출력 결과:  

KeyError: 'english'
🔥 오류 발생! "english"는 course_counts에 없기 때문에 KeyError가 발생하는 거야.

In [17]:
# 모든 학생에게 강의실 배정
# apply() 함수 개념: apply()는 데이터프레임의 행(axis=1) 또는 열(axis=0)을 순회하면서, 특정 함수를 적용
# df["새로운 컬럼"] = df["기존 컬럼"].apply(함수)
df["room assignment"] = df["course name"].apply(assign_room)

In [18]:
# "not allowed" 학생들의 강의실을 "not assigned"로 변경
df.loc[df["status"] == "not allowed", "room assignment"] = "not assigned"

# 과제 3

## 과제 조건 

### 이전 과제에서 강의실 크기에 따라 room assignment column을 만들어 주었습니다.

이제 이 room assignment에 따라 강의실 이름을 붙여주려고 합니다.

아래 세 가지 조건을 만족하도록 코드를 작성하세요.

1. 같은 크기의 강의실이 필요한 과목에 대해 알파벳 순서대로 방 번호를 배정하세요.

 예를 들어 "Auditorium"이 필요한 과목으로 "arts", "commerce", "science" 세 과목이 있다면, "arts"는 "Auditorium-1", "commerce"는  "Auditorium-2", "science"는 "Auditorium-3" 순서로 방 배정이 되어야 합니다.

방 번호에 room 은 포함되지 않습니다. 아래 스크린샷을 참고하여 작성해주세요.

2. status column이 "not allowed"인 수강생은 room assignment column을 그대로 "not assigned"로 남겨둡니다. "not allowed" 인 수강생의 room assignment 상태가 변경되지 않도록 유의해주세요.

3. room assignment column의 이름을 room number로 바꿔주세요.

In [19]:
# 다시 과목별 인원과 "allowed"의 과목들만 바꾸도록 조건 생성해주기 
c1 = df["status"] == "allowed"
course_counts = df[df["status"] == "allowed"]["course name"].value_counts()

In [20]:
# 각 강의실 규모에 해당되는 리스트 만들어주기 

auditorium_list = list(course_counts[course_counts >= 80].index)
large_room_list = list(course_counts[(course_counts >= 40) & (course_counts < 80)].index)
medium_room_list = list(course_counts[(course_counts >= 15) & (course_counts < 40)].index)
small_room_list = list(course_counts[(course_counts >= 5) & (course_counts < 15)].index)

print(auditorium_list)
print(large_room_list)
print(medium_room_list)
print(small_room_list)

['arts', 'science', 'commerce']
['english', 'education']
['management', 'nursing', 'chemistry', 'computer', 'computer science', 'civil engineering', 'history', 'mechanical engineering', 'physics', 'civil', 'economics', 'mathematics', 'coumputer science', 'mechanical', 'engineering', 'botany', 'cse', 'mbbs', 'zoology', 'computer application', 'law', 'electrical', 'chemical engineering', 'general']
['sociology', 'sanskrit', 'computer applications', 'hindi', 'information technology', 'computer engineering', 'electronics & communication engg', 'ee', 'urdu', 'business administration', 'bzc', 'philosophy', 'yoga', 'bengali', 'marathi', 'political science', 'electrical engineering', 'maths', 'computer science & engineering', 'finance', 'technical', 'ece', 'me', 'electrical & electronics engineering', 'biotechnology', 'computer science and engineering', 'ug', 'physiotherapy', 'electronics', 'bio-chemistry', 'general medicine', 'electronics and communication engineering', 'anatomy', 'bio chemis

In [21]:
# 각 강의실 규모에 해당되는 과목 리스트에 강의실 이름 붙이기

for i in range(len(auditorium_list)):
    df.loc[(df["course name"] == sorted(auditorium_list)[i]) & c1, "room assignment"] = "Auditorium-" + str(i+1)
    
for i in range(len(large_room_list)):
    df.loc[(df["course name"] == sorted(large_room_list)[i]) & c1, "room assignment"] = "Large-" + str(i+1)
    
for i in range(len(medium_room_list)):
    df.loc[(df["course name"] == sorted(medium_room_list)[i]) & c1, "room assignment"] =  "Medium-" + str(i+1)
    
for i in range(len(small_room_list)):
    df.loc[(df["course name"] == sorted(small_room_list)[i]) & c1, "room assignment"] = "Small-" + str(i+1)

### 이 코드는  과목별 리스트들의 각 항목에 대해, 해당 항목이 "course name" 열에 있는 행을 찾아, 그 행들의 "room assignment" 열을 "Auditorium-1", "Auditorium-2"와 같은 형식으로 수정하는 코드입니다. 이때, c1 조건도 추가되어야 합니다.

예시:
`auditorium_list = ["Math", "Physics", "Chemistry"]`  
**df["course name"]** 에 "Math", "Physics", "Chemistry"와 같은 값이 있다고 가정하면, 정렬된 "auditorium_list"의 첫 번째 항목인 "Chemistry"와 일치하는 행을 찾습니다.  
그 행의 "room assignment" 열을 "Auditorium-1"로 설정합니다.  
이 과정을 i 값을 증가시키며 반복합니다.  
따라서 각 과목에 대해 정렬된 auditorium_list의 순서에 맞게 "room assignment" 값을 할당하는 작업입니다.  

In [22]:
# column 이름 바꾸기
df.rename(columns={"room assignment": "room number"}, inplace = True)

In [24]:
df

,id,year,course name,status,room number
0,2777729,1,information technology,not allowed,not assigned
1,2777730,2,science,allowed,Auditorium-3
2,2777765,1,arts,allowed,Auditorium-1
3,2777766,2,arts,allowed,Auditorium-1
4,2777785,1,mba,allowed,Small-34
...,...,...,...,...,...
1995,2796805,3,computer application,allowed,Medium-7
1996,2796812,1,nursing,allowed,Medium-22
1997,2796813,2,nursing,allowed,Medium-22
1998,2796814,3,nursing,allowed,Medium-22
